# Harmonization Ablation Before CNN

The cross-dataset baseline showed strong domain shift. This notebook tests lightweight representations that may reduce sensor-axis and scale differences before a deep model is introduced:

1. all 18 channels in physical units;
2. accelerometer channels only;
3. orientation-invariant acceleration magnitudes;
4. per-window dynamic acceleration shape features.

The comparison uses the same participant-level aggregation and both cross-dataset directions.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, roc_auc_score
from sklearn.preprocessing import StandardScaler

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
windows = np.load(PROCESSED / 'validated_gait_windows_float32.npy', mmap_mode='r')
metadata = pd.read_csv(PROCESSED / 'validated_window_metadata.csv')
metadata['label_binary'] = metadata['label'].map({'healthy': 0, 'stroke': 1}).astype(int)
print('Windows:', windows.shape)


Windows: (18511, 500, 18)


In [2]:
ACCEL_INDICES = [0, 1, 2, 6, 7, 8, 12, 13, 14]
PLACEMENT_INDICES = [(0, 1, 2), (6, 7, 8), (12, 13, 14)]

def summarize(values):
    return np.concatenate([values.mean(axis=1), values.std(axis=1), np.sqrt(np.mean(values ** 2, axis=1))], axis=1)


def build_features(window_array, variant, batch_size=1024):
    chunks = []
    for start in range(0, len(window_array), batch_size):
        batch = np.asarray(window_array[start:start + batch_size], dtype=np.float32)
        accel = batch[:, :, ACCEL_INDICES]
        if variant == 'raw_all_18ch':
            values = batch
            features = summarize(values)
        elif variant == 'raw_accel_9ch':
            features = summarize(accel)
        elif variant == 'accel_magnitude_3ch':
            magnitudes = np.stack([np.linalg.norm(batch[:, :, idx], axis=2) for idx in PLACEMENT_INDICES], axis=2)
            features = summarize(magnitudes)
        elif variant == 'dynamic_accel_shape':
            dynamic = accel - accel.mean(axis=1, keepdims=True)
            scale = dynamic.std(axis=1, keepdims=True) + 1e-6
            normalized = dynamic / scale
            diff_rms = np.sqrt(np.mean(np.diff(normalized, axis=1) ** 2, axis=1))
            quantiles = np.percentile(normalized, [10, 25, 50, 75, 90], axis=1).transpose(1, 2, 0).reshape(len(batch), -1)
            features = np.concatenate([dynamic.std(axis=1), diff_rms, quantiles], axis=1)
        else:
            raise ValueError(variant)
        chunks.append(features)
    return np.concatenate(chunks, axis=0)


def evaluate_cross_dataset(features):
    rows = []
    for train_dataset, test_dataset in [('voisard_2025', 'felius_2024'), ('felius_2024', 'voisard_2025')]:
        train_mask = metadata['dataset_id'].eq(train_dataset)
        test_mask = metadata['dataset_id'].eq(test_dataset)
        train_indices = metadata.index[train_mask].to_numpy()
        test_indices = metadata.index[test_mask].to_numpy()
        scaler = StandardScaler()
        train_x = scaler.fit_transform(features[train_indices])
        test_x = scaler.transform(features[test_indices])
        train_y = metadata.loc[train_mask, 'label_binary'].to_numpy()
        counts = metadata.loc[train_mask].groupby('participant_key').size()
        weights = metadata.loc[train_mask, 'participant_key'].map(1.0 / counts).to_numpy()
        weights = weights / weights.mean()
        model = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
        model.fit(train_x, train_y, sample_weight=weights)
        probabilities = model.predict_proba(test_x)[:, 1]
        test_frame = metadata.loc[test_mask, ['participant_key', 'label_binary']].copy()
        test_frame['probability'] = probabilities
        participant_frame = test_frame.groupby(['participant_key', 'label_binary'], as_index=False)['probability'].mean()
        y_true = participant_frame['label_binary'].to_numpy()
        y_prob = participant_frame['probability'].to_numpy()
        rows.append({
            'train_dataset': train_dataset,
            'test_dataset': test_dataset,
            'test_participants': len(participant_frame),
            'balanced_accuracy': balanced_accuracy_score(y_true, (y_prob >= 0.5).astype(int)),
            'roc_auc': roc_auc_score(y_true, y_prob),
        })
    return pd.DataFrame(rows)


In [3]:
all_results = []
for variant in ['raw_all_18ch', 'raw_accel_9ch', 'accel_magnitude_3ch', 'dynamic_accel_shape']:
    features = build_features(windows, variant)
    assert np.isfinite(features).all()
    result = evaluate_cross_dataset(features)
    result.insert(0, 'variant', variant)
    result.insert(1, 'feature_count', features.shape[1])
    all_results.append(result)
    print(variant)
    print(result.round(3).to_string(index=False))
    print()

results = pd.concat(all_results, ignore_index=True)
results.to_csv(PROCESSED / 'harmonization_ablation_results.csv', index=False)
print('Saved:', PROCESSED / 'harmonization_ablation_results.csv')


raw_all_18ch
     variant  feature_count train_dataset test_dataset  test_participants  balanced_accuracy  roc_auc
raw_all_18ch             54  voisard_2025  felius_2024                163               0.59    0.837
raw_all_18ch             54   felius_2024 voisard_2025                121               0.50    0.148



raw_accel_9ch
      variant  feature_count train_dataset test_dataset  test_participants  balanced_accuracy  roc_auc
raw_accel_9ch             27  voisard_2025  felius_2024                163              0.603    0.726
raw_accel_9ch             27   felius_2024 voisard_2025                121              0.142    0.146



accel_magnitude_3ch
            variant  feature_count train_dataset test_dataset  test_participants  balanced_accuracy  roc_auc
accel_magnitude_3ch              9  voisard_2025  felius_2024                163              0.736    0.863
accel_magnitude_3ch              9   felius_2024 voisard_2025                121              0.663    0.826



dynamic_accel_shape
            variant  feature_count train_dataset test_dataset  test_participants  balanced_accuracy  roc_auc
dynamic_accel_shape             63  voisard_2025  felius_2024                163              0.527    0.858
dynamic_accel_shape             63   felius_2024 voisard_2025                121              0.639    0.704

Saved: C:\Users\frank\Documents\MR-ICT Review Paper\data\processed\harmonization_ablation_results.csv


## Decision gate

Select the representation that improves the weaker cross-dataset direction without destroying the stronger direction. That representation becomes the input contract for the first CNN. If no representation is robust, the project should recruit more harmonized data or treat the first model as a within-dataset pilot rather than a general classifier.